# Agentic AI: Week 1 Lab
# Setup, Prompting & Your First Agent


---
## Part 1: Environment and Imports

We use the OpenAI Python SDK as our client library because most LLM providers (Groq, Ollama, our university API) expose an OpenAI-compatible API.

In [6]:
# Install the OpenAI SDK (works with any OpenAI-compatible API)
!uv add openai

Resolved 16 packages in 224ms                                        
⠙ Preparing packages... (0/7)                                                   
⠙ Preparing packages... (0/7)--------------     0 B/81.47 KiB           
⠙ Preparing packages... (0/7)-------------- 14.83 KiB/81.47 KiB         
⠙ Preparing packages... (0/7)-------------- 14.83 KiB/81.47 KiB         
truststore           ------------------------------     0 B/18.57 KiB
⠙ Preparing packages... (0/7)-------------- 14.83 KiB/81.47 KiB         
truststore           ------------------------------ 14.76 KiB/18.57 KiB
⠙ Preparing packages... (0/7)-------------- 14.83 KiB/81.47 KiB         
truststore           ------------------------------ 14.76 KiB/18.57 KiB
⠙ Preparing packages... (0/7)-------------- 30.83 KiB/81.47 KiB         
truststore           ------------------------------ 14.76 KiB/18.57 KiB
⠙ Preparing packages... (0/7)2m------------ 46.83 KiB/81.47 KiB         
truststore           ------------------------------ 

In [7]:
# You can skip this part if you just put the API key as plain text
import os
from pathlib import Path

# Load .env file if it exists (no extra dependencies needed)
env_path = Path(".env")
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, _, value = line.partition("=")
            os.environ[key.strip()] = value.strip()


---
## Part 2: Set Up Provider

The last of these three cells you run will be the model used in the following cells.


###  Apex Orion (if available)

Our Apex-hosted Mango1.5 (based on Qwen3.5 9B): https://orion.cmkl.ai/

In [8]:
from openai import OpenAI


#ORION_API_KEY = "sk-..."
ORION_API_KEY = os.environ.get("ORION_API_KEY", "")

ORION_BASE_URL = "https://api.orion.cmkl.ai/v1"

client = OpenAI(
    api_key=ORION_API_KEY if ORION_API_KEY else "not-needed",
    base_url=ORION_BASE_URL,
)

MODEL = "Mango 1.5-9B"
PROVIDER = "Orion"

print(f"Using {MODEL} via {PROVIDER}")


Using Mango 1.5-9B via Orion


### Groq

1. Go to [console.groq.com](https://console.groq.com) and create a free account
2. Go to **API Keys** and create a new key
3. Copy the key and paste it belor or in your `.env` file

In [ ]:
from openai import OpenAI


#GROQ_API_KEY = "gsk_..."
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")

client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",
)

MODEL = "openai/gpt-oss-20b"
PROVIDER = "Groq"

print(f"Using {MODEL} via {PROVIDER}")


### Ollama (local Mac/Linux users)

If you have a Mac (Apple Silicon) with enough RAM or a Linux machine with a decent GPU and enough VRAM, you can run models locally with [Ollama](https://ollama.com).

**Setup:**
1. Download and install Ollama from [ollama.com](https://ollama.com)
2. Open a terminal and run: `ollama pull gpt-oss:20b`
3. Wait for the download to finish (~12 GB)
4. Ollama runs a local API server automatically on port 11434


In [ ]:
from openai import OpenAI


client = OpenAI(
    api_key="ollama",  # Ollama doesn't need a real key but the SDK requires something
    base_url="http://localhost:11434/v1",
)

MODEL = "gpt-oss:20b"  # must match the model you pulled with `ollama pull`
PROVIDER = "Ollama (local)"

print(f"Using {MODEL} via {PROVIDER}")


---
## Part 3: Test Your Connection

Let's make sure everything works with a simple call.

In [9]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "Say hello and tell me which model you are."}
    ],
)

print(response.choices[0].message.content)




Hello! I am a large language model, trained by Google.


If you see a response, you're good to go! If you get an error:
- **401 / Unauthorized**: Check your API key
- **Connection refused**: For Ollama, make sure it's running (`ollama serve` in terminal)
- **Rate limit error**: Wait a moment and try again
- **Model not found**: Check the MODEL variable matches what's available

---
## Part 4: Understanding the Chat API

Every call uses the same structure: a list of **messages**, each with a **role** and **content**.

The three roles:
- `system`: sets the behavior, personality, and constraints of the model
- `user`: the human's input
- `assistant`: the model's response (used for conversation history)

In [10]:
def chat(user_message, system_prompt=None, temperature=0.7):
    """Simple helper to send a message and get a response."""
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": user_message})

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=temperature,
    )
    return response.choices[0].message.content

# Test it
print(chat("What is 2 + 2?"))




2 + 2 = 4


---
## Part 5: System Prompts

The system prompt is how you control **what** the model does and **how** it behaves. For agents, this is where the agent's capabilities, constraints, and personality are defined.

### Exercise 1: Bad prompt vs. good prompt

Let's ask the model to extract structured data from a messy text. First with a lazy prompt, then with a well-crafted one.

In [11]:
messy_text = """
Hey so I talked to the client today, they want the website done by March 15th,
budget is around 50k but might go up to 75k if we include the mobile app.
Main contact is Sarah Chen, sarah.c@techcorp.com. They also mentioned
they need it to integrate with Salesforce and their existing PostgreSQL database.
Oh and they want it in React, no Angular.
"""

# Bad prompt: vague, no structure
bad_response = chat(
    f"Get the important info from this: {messy_text}"
)
print("=== BAD PROMPT ===")
print(bad_response)


=== BAD PROMPT ===


Here is the key information extracted from the update:

*   **Project Deadline:** March 15th
*   **Budget:** $50k (base) to $75k (if mobile app is included)
*   **Tech Stack:** React (specifically requested; no Angular)
*   **Integrations:** Salesforce and PostgreSQL database
*   **Main Contact:** Sarah Chen (sarah.c@techcorp.com)


In [12]:
# Good prompt: specific role, structured output, clear constraints
good_response = chat(
    user_message=messy_text,
    system_prompt="""You are a project intake assistant. Extract structured information
from client communications.

Always respond in this exact JSON format (no other text):
{
  "client_name": "company or person name",
  "contact_email": "email address",
  "deadline": "YYYY-MM-DD",
  "budget_min": number,
  "budget_max": number,
  "requirements": ["list of technical requirements"],
  "constraints": ["list of explicit constraints or preferences"]
}

If a field is unknown, use null. Do not make up information.""",
    temperature=0.0,
)

print("=== GOOD PROMPT ===")
print(good_response)


=== GOOD PROMPT ===


{
  "client_name": "TechCorp",
  "contact_email": "sarah.c@techcorp.com",
  "deadline": "2025-03-15",
  "budget_min": 50000,
  "budget_max": 75000,
  "requirements": [
    "Integrate with Salesforce",
    "Integrate with existing PostgreSQL database",
    "Develop using React",
    "No Angular"
  ],
  "constraints": [
    "Must be completed by March 15, 2025",
    "Budget range $50,000 to $75,000",
    "Technology stack: React (no Angular)",
    "Integrate with Salesforce and existing PostgreSQL database",
    "Mobile app optional"
  ]
}


### Discussion

- What differences do you notice?
- Could you reliably **parse** the bad prompt's output in code?
- Why does this matter for agents?

### Exercise 2: Fun with system prompts

Same question, different system prompts: watch how the behavior changes completely.

In [13]:
question = "Should I learn Rust or Go for backend development?"

personas = {
    "No system prompt": None,
    "Concise expert": "You are a senior software engineer. Answer in at most 3 sentences.",
    "Socratic teacher": "You are a computer science teacher. Never give a direct answer. Instead, ask questions that help the student figure it out themselves.",
    "Opinionated blogger": "You are a tech blogger known for strong, controversial opinions. Be bold and pick a side. Keep it under 4 sentences.",
}

for name, prompt in personas.items():
    print(f"\n{'='*60}")
    print(f"PERSONA: {name}")
    print(f"{'='*60}")
    print(chat(question, system_prompt=prompt))
    


PERSONA: No system prompt


Both are excellent choices for backend development, but they serve different philosophies and use cases. The "right" choice depends heavily on what kind of developer you want to be, the performance requirements of your projects, and your career goals.

Here is a breakdown to help you decide.

---

### 1. Go (Golang)
**The "Get Things Done" Language**

Go is the king of backend web development for the past decade. It was designed by Google specifically for cloud-native development, microservices, and infrastructure tools (like Docker and Kubernetes).

**Pros:**
*   **Simplicity & Speed:** The syntax is simple and concise. You can write a backend service in hours rather than days.
*   **Fast Compilation:** Go compiles to machine code instantly. This makes development and deployment significantly faster than other languages.
*   **Concurrency:** Go has built-in concurrency primitives (Goroutines and Channels) that make writing network-intensive code easy and r

### Exercise 3: Your turn

Write a system prompt that makes the model behave as a **code reviewer**. It should:
- Point out bugs and issues
- Suggest improvements
- Rate the code on a scale of 1-10
- Respond in a structured format

Test it with the code snippet below.

In [15]:
code_to_review = """
def get_user_data(user_id):
    import requests
    data = requests.get(f"http://api.example.com/users/{user_id}").json()
    password = data['password']
    return {"name": data['name'], "email": data['email'], "password": password}
"""

# TODO: Write your system prompt here
your_system_prompt = """
You are a senior QA engineer. Review the provided code for bugs and issues. 
Suggest how we can improve the code. Also rate the code on a scale of 1-10.
Responnd in a structured format with the following fields:
{
  "bugs": ["list of bugs or issues found"],
  "improvements": ["list of suggested improvements"],
  "rating": number (1-10)
}
"""

print(chat(f"Review this code:\n{code_to_review}", system_prompt=your_system_prompt))




{
  "bugs": [
    "Critical Security Vulnerability: The function retrieves and returns the user's password. This is a severe security breach; sensitive data like passwords should never be returned in API responses or logs.",
    "Lack of Error Handling: The code assumes the API call always succeeds and the response always contains 'name', 'email', and 'password'. It will crash with a `KeyError` or `requests.exceptions.RequestException` if the API returns an error or lacks these fields.",
    "Inefficient Import Placement: The `import requests` statement is inside the function. It is best practice to import modules at the top of the script for performance and readability.",
    "Hardcoded Configuration: The API URL is hardcoded. This makes the code rigid and difficult to maintain or deploy in different environments."
  ],
  "improvements": [
    "Remove Password from Return: Eliminate the retrieval and return of the password to ensure data privacy.",
    "Add Error Handling: Wrap the 

### Exercise 4: Have fun!

Change the system prompt to anything you want. Tell it to behave like a famous person, talk like a poet, act like a specific age group, ...